In [17]:
import math
import cv2
import numpy as np
import gymnasium as gym

from time import sleep
from gymnasium import spaces

from typing import Dict, Any, Sequence, Tuple, Optional

from importnb import Notebook
with Notebook():
    from LabTrajectory import simulate_viewport_with_tiles

import sys
sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')

from Model.TileIndexer import TileIndexer
from Common.Utils import zipf

In [18]:
def previousTilePrefetcher(step, i, users_viewport_tiles, n) -> np.ndarray:
    step -= 1

    action = np.zeros(n*n, dtype=int)
    
    if step < 0: return action
    
    for tile in users_viewport_tiles[i][step]:
        tx, ty = tile[0], tile[1]
        if 0 > tx or tx >= n or 0 > ty or ty >= n:
            continue
        action[tx * n + ty] = 1
    return action

In [19]:
# ---------- Helper functions implementing formulas from the paper ----------
def compute_td_u(
    B_pu: float, 
    gamma_pu: float
) -> float:
    """
    TD,U = 1 / (Bp,u * log2(1 + gamma_pu))
    B_pu: bandwidth (bytes/sec) for DU p to user u (or effective bandwidth)
    gamma_pu: average SNR
    returns average latency (seconds) per byte
    """
    # prevent division by zero or log2(1+gamma)=0
    denom = B_pu * np.log2(1.0 + max(gamma_pu, 1e-12))
    if denom <= 0:
        return np.inf
    return 1.0 / denom

def compute_tm_u(
    td_u: float, 
    R_M_D: float, 
    U: int
) -> float:
    """
    TM,U = TD,U + 1 / (R_M,D / U) = TD,U + U / R_M,D
    R_M_D: total data rate from MEC to DU (bytes/sec)
    U: number of users sharing link equally
    returns average latency (seconds) per byte
    """
    if R_M_D <= 0 or U <= 0:
        return np.inf
    return td_u + (U / R_M_D)

def compute_tc_u(
    tm_u: float, 
    R_C_M: float, 
    U: int
) -> float:
    """
    TC,U = TM,U + 1 / (R_C,M / U) = TM,U + U / R_C,M
    R_C_M: total data rate from Cloud to MEC (bytes/sec)
    """
    if R_C_M <= 0 or U <= 0:
        return np.inf
    return tm_u + (U / R_C_M)

def compute_ttc(
    rhoT_p: Sequence[float],
    lambda_p: Sequence[float],
    eta: float,
    mu: float
) -> float:
    """
    Ttc = (sum_p rhoT_p * lambda_p / eta) / (mu - sum_p lambda_p)
    Conditions: mu - sum(lambda_p) > 0
    rhoT_p, lambda_p arrays must be same length P
    eta: average data size of computation tasks (bytes)
    mu: service rate of MEC server (bytes/sec of processing capacity)
    returns avg computation latency per byte (seconds/byte)
    """
    rhoT_p = np.asarray(rhoT_p, dtype=float)
    lambda_p = np.asarray(lambda_p, dtype=float)
    if eta <= 0:
        raise ValueError("eta must be > 0")
    denom = mu - np.sum(lambda_p)
    if denom <= 0:
        return np.inf
    numerator = np.sum(rhoT_p * lambda_p / eta)
    return numerator / denom

In [20]:
class LatencyModel:
    """
    LatencyModel computes per-tile latency and total latency for requests
    using the formulas from the paper (TD,U, TM,U, TC,U, Ttc and D matrix).

    P: number of DUs (p in paper)
    U: number of users
    R_M_D: total MEC->DU data rate (bytes/sec)
    R_C_M: total Cloud->MEC data rate (bytes/sec)
    mu: MEC service rate (bytes/sec) for transcoding
    eta: average computation task size (bytes)
    B_pu_matrix: shape (P, U) bandwidths from DU p to user u (bytes/sec). If None, set to 1e6
    gamma_pu_matrix: shape (P, U) SNRs. If None, set to 10 (10 linear)
    rhoT_p: length P list of proportion of transcoding tasks requested at CU
    lambda_p: length P list of arrival rates (requests/sec) from each DU
    """
    def __init__(
        self,
        P: int,
        U: int,
        R_M_D: float,
        R_C_M: float,
        mu: float,
        eta: float,
        B_pu_matrix: np.ndarray,
        gamma_pu_matrix: np.ndarray,
        rhoT_p: Sequence[float],
        lambda_p: Sequence[float],
        du_fixed_delay: float,
        mec_fixed_delay: float,
        cloud_fixed_delay: float
    ):
        self.P = P
        self.U = U
        self.R_M_D = float(R_M_D)
        self.R_C_M = float(R_C_M)
        self.mu = float(mu)
        self.eta = float(eta)

        self.du_fixed_delay = float(du_fixed_delay)
        self.mec_fixed_delay = float(mec_fixed_delay)
        self.cloud_fixed_delay = float(cloud_fixed_delay)

        self.B_pu = np.asarray(B_pu_matrix, dtype=float).reshape(P, U)
        self.gamma_pu = np.asarray(gamma_pu_matrix, dtype=float).reshape(P, U)
        self.rhoT_p = np.asarray(rhoT_p, dtype=float)
        self.lambda_p = np.asarray(lambda_p, dtype=float)

        # precompute TD,U
        self.TD_U = np.zeros((P, U), dtype=float)
        for p in range(P):
            for u in range(U):
                self.TD_U[p, u] = compute_td_u(
                    self.B_pu[p, u], 
                    self.gamma_pu[p, u]
                )

    def time_vector(self, p: int, u: int) -> Tuple[float, float, float, float, float]:
        td_u = self.TD_U[p, u]
        tm_u = compute_tm_u(td_u, self.R_M_D, self.U)
        tc_u = compute_tc_u(tm_u, self.R_C_M, self.U)
        ttc = compute_ttc(self.rhoT_p, self.lambda_p, self.eta, self.mu)
        
        return td_u, tm_u, tc_u, tm_u + ttc, tm_u + ttc

    def tile_latency(
        self, 
        p: int, 
        u: int, 
        tile_size_bytes: float, 
        events: Dict[str, int]
    ) -> float:
        """
        events dict must contain binary flags for:
         - 'alpha_p_u', 'alpha_M_u', 'alpha_C_u', 'beta_p_u', 'beta_M_u'
        Order corresponds to the 5 columns in the paper's D matrix.
        Returns latency in seconds for that tile (per tile_size_bytes).
        """
        T_vec = np.array(
            self.time_vector(p, u), 
            dtype=float
        )
        ES_vec = np.array([
            events.get("alpha_p_u", 0),
            events.get("alpha_M_u", 0),
            events.get("alpha_C_u", 0),
            events.get("beta_p_u", 0),
            events.get("beta_M_u", 0),
        ], dtype=float)
        
        per_byte_latency = float(np.dot(ES_vec, T_vec) * tile_size_bytes)

        # add fixed propagation delay(s) - only once per tile when DU or MEC are involved
        fixed = 0.0
        # if served from DU (column alpha_p_u) -> add DU fixed delay
        if events.get("alpha_p_u", 0) == 1 or events.get("beta_p_u", 0) == 1:
            fixed += self.du_fixed_delay
        # if served via MEC (alpha_M_u or beta_M_u) -> add MEC fixed delay
        if events.get("alpha_M_u", 0) == 1 or events.get("beta_M_u", 0) == 1:
            fixed += self.mec_fixed_delay
        # alpha_C_u (cloud) may also include additional fixed delays you can add as needed
        if events.get("alpha_C_u", 0) == 1:
            fixed += self.cloud_fixed_delay

        return per_byte_latency + fixed

    def total_request_latency(
        self, 
        p: int, 
        u: int, 
        tiles: Sequence[Dict[str, Any]]
    ) -> Tuple[float, Sequence[Tuple[str, float]]]:
        tile_latencies = []
        total = 0.0
        
        for tile in tiles:
            sz = float(tile["size"])
            events = tile.get("events", {})
            tid = tile.get("tile_id", None)

            lat = self.tile_latency(p, u, sz, events)
            tile_latencies.append((tid, lat))
            total += lat

        return total, tile_latencies

    def compute_latency_for_request(self, request) -> Dict[str, Any]:
        p = request.get("du_id", 0)
        u = request.get("user_id", 0)
        tiles = request.get("tiles", [])

        total_latency, tile_latencies = self.total_request_latency(
            p, 
            u, 
            tiles
        )
        
        return {
            "total_latency": total_latency,
            "tile_latencies": tile_latencies
        }
    
    def user_link_throughput(
        self, 
        p: int, 
        u: int
    ) -> Dict[str, float]:
        """
        Returns per-user throughput (bytes/sec) for each stage:
        - 'du': wireless DU->user
        - 'mec': DU->user + MEC->DU (shared)
        - 'cloud': DU->user + MEC->DU + Cloud->MEC (shared)
        """
        td_u = self.TD_U[p, u]                         # sec/byte
        tm_u = compute_tm_u(td_u, self.R_M_D, self.U)  # sec/byte
        tc_u = compute_tc_u(tm_u, self.R_C_M, self.U)  # sec/byte

        def inv(x): return 0.0 if not np.isfinite(x) or x <= 0 else (1.0 / x)

        return {
            "du": inv(td_u),        # bytes/sec over DU->user link
            "mec": inv(tm_u),       # bytes/sec including MEC->DU
            "cloud": inv(tc_u)      # bytes/sec including Cloud->MEC
        }

    def request_effective_throughput(
        self, 
        p: int, 
        u: int, 
        tiles: Sequence[Dict[str, Any]]
    ) -> float:
        """
        Effective throughput for a request = total_bytes / total_time.
        Uses tile_latency() to accumulate time over all tiles.
        """
        total_bytes = 0.0
        total_time = 0.0
        for tile in tiles:
            sz = float(tile["size"])
            events = tile.get("events", {})
            total_bytes += sz
            total_time += self.tile_latency(p, u, sz, events)
        
        if total_time <= 0 or not np.isfinite(total_time):
            return 0.0
        
        return total_bytes / total_time

In [21]:
class UserTileRequestEvents:
    def __init__(
        self,
        n: int = 4,
        n_users: int = 1,
        n_layers: int = 1,
        n_tiles: int = 4,
        users_viewport_tiles: Optional[Sequence[Any]] = [],
        requested_videos: Optional[Sequence[Any]] = []
    ):
        self.n = n
        self.step_count = 0
        self.n_users = n_users
        self.n_layers = n_layers
        self.n_tiles = n_tiles
        self.users_viewed_tiles = users_viewport_tiles
        self.videos_requests = requested_videos
    
    def get_request_for_user(
        self, 
        step: int, 
        u_id: int, 
        p_id: int,
        by_video
    ) -> Dict[str, Any]:
        # if size is in MB
        size_bytes = (int(15) * 1024 * 1024) / self.n_tiles
        
        # For demonstration, return a fixed request structure
        video = self.videos_requests[u_id]
        tiles = []
        for x, y in self.users_viewed_tiles[u_id][step]:
            
            if x < 0 or x >= self.n or y < 0 or y >= self.n:
                continue

            tile = {
                "tile_id": y * self.n + x,
                "size": size_bytes,
                "events": {
                    "alpha_p_u": 1 if by_video[video][y * self.n + x] > 0 else 0, 
                    "alpha_M_u": 0, 
                    "alpha_C_u": 0 if by_video[video][y * self.n + x] > 0 else 1, 
                    "beta_p_u": 0, 
                    "beta_M_u": 0   
                }
            }
            tiles.append(tile)
        
        return {
            "seq": step,
            "u": u_id,
            "p": p_id, 
            "video": video,
            "tiles": tiles
        }

    def step(self, by_video):
        p = 0  # assuming DU id 0 for simplicity
        reqs = [
            self.get_request_for_user(self.step_count, i, p, by_video)
            for i in range(self.n_users)
        ]

        self.step_count += 1
        return reqs

    def reset(self, **kwargs):
        return np.array([0.0]), {}

In [22]:
class PrefetchingCacheEnv:
    def __init__(
        self, 
        n: int = 4,
        n_users: int = 1,
        num_tiles: int = 16,
        cache_capacity: int = 10,
        users_viewport_tiles: np.ndarray = [],
        videos_requests: list = [],
    ):
        self.capacity = self.cache_capacity = cache_capacity

        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = 0.5
        self.lam = 3.7183

        self.step_count = 0
        self.n = n
        self.n_users = n_users
        self.num_tiles = num_tiles
        self.users_viewport_tiles = users_viewport_tiles
        self.videos_requests = videos_requests

    def step(self, action):
        ### STEP 1: prefetching segments in the cache based on action taken by agent ###
        # Aggregate prefetching actions by video and tile
        by_video = {}
        for user_action in action:
            vid = user_action.get('video', None)
            tiles = user_action.get('tiles', None)

            if vid not in by_video:
                by_video[vid] = np.zeros(self.num_tiles, dtype=int)
            
            by_video[vid] += (tiles > 0)

        video_tile_counts = []
        for vid, arr in by_video.items():
            for tile_idx, count in enumerate(arr):
                video_tile_counts.append((vid, tile_idx, count))

        # Sort by count descending
        video_tile_counts = sorted(video_tile_counts, key=lambda x: x[2], reverse=True)

        n_prefetch = len(video_tile_counts)
        if n_prefetch > self.capacity:
            to_drop = int(n_prefetch - self.capacity)
            dropped = video_tile_counts[-to_drop:]

            for vid, tile_idx, count in dropped:
                by_video[vid][tile_idx] = 0
    
        self.step_count += 1

        return by_video

In [23]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    def __init__(
        self, 
        users_env: None,
        cache_env: None,
        latency_model: LatencyModel, 
        fetch_request_callable=None
    ):
        super().__init__()
        self.users_env = users_env
        self.cache_env = cache_env
        self.latency_model = latency_model
        self.fetch_request_callable = fetch_request_callable

    def step(self, action):
        ### STEP 1: prefetching segments in the cache based on action taken by agent ###
        by_video = self.cache_env.step(action)
        print("By video prefetching:", by_video)

        ### STEP 2: process user requests and compute latencies ###
        reqs = self.users_env.step(by_video)
        print("User requests:", reqs)

        ### STEP 3: compute latencies for each request ###
        total_latency = 0.0
        for req in reqs:
            p = req.get("p", 0)
            u = req.get("u", 0)
            tiles = req.get("tiles", [])
            
            req_latency, tile_latencies = self.latency_model.total_request_latency(
                p, 
                u, 
                tiles
            )

            total_latency += req_latency
            print(f"Request for user {u}, DU {p}, total latency: {req_latency:.4f} sec")

            tp = self.latency_model.user_link_throughput(p=p, u=u)
            print(tp["du"], tp["mec"], tp["cloud"])

            req_tp = self.latency_model.request_effective_throughput(p=p, u=u, tiles=req["tiles"])
            print(req_tp, "bytes/sec")
        
        
        return {}, 1.0, False, {}

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

    def render(self, mode="human"):
        return self.env.render(mode)

In [ ]:
if __name__ == "__main__":
    n_episodes = 500
    n_steps = 60
    total_videos = 100
    n_frames = 60
    n_users = 12
    n = 4

    users_viewport_tiles = []
    for f in range(n_users):
        yaw, pitch, tiles_per_frame, tiles_array = simulate_viewport_with_tiles(
            num_steps=n_frames,
            n=n,
            fov_yaw=90,
            fov_pitch=50,
            damping=0.99,
            step_size=5.0,
            start_yaw=180,
            start_pitch=0
        )

        users_viewport_tiles.append(tiles_per_frame)
    
    requested_videos = zipf(n_users, total_videos=total_videos, alpha=1.0)

    users_env = UserTileRequestEvents(
        n_users=n_users,
        n_tiles=n*n,
        users_viewport_tiles=users_viewport_tiles,
        requested_videos=requested_videos
    )

    cache_env = PrefetchingCacheEnv(
        num_tiles=n*n,
        n_users=n_users,
        cache_capacity=100,
        users_viewport_tiles=users_viewport_tiles,
        videos_requests=requested_videos 
    )

    # Create latency model (replace numbers with your real config)
    P = 1; U = n_users
    # Provide (P, U) arrays instead of (1,1)
    B_pu_matrix = np.full((P, U), 40e6, dtype=float)       # 320 Mbps -> 40e6 B/s
    gamma_pu_matrix = np.full((P, U), 5.0, dtype=float)    # SNRs

    lat_model = LatencyModel(
        P=P, 
        U=U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=B_pu_matrix,
        gamma_pu_matrix=gamma_pu_matrix,
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        users_env=users_env, 
        cache_env=cache_env, 
        latency_model=lat_model
    )


    total_reward = 0.0
    for step in range(n_steps):
        actions = [
            {
                'user': i,
                'video': requested_videos[i],
                'tiles': previousTilePrefetcher(step, i, users_viewport_tiles, n)
            } for i in range(n_users)
        ]

        obs, reward, _, info = env.step(actions)

        total_reward += float(reward)

    

    # wrapped = LatencyWrapper(users_env, latency_model=lat_model)

    # obs, reward, done, info = wrapped.step(0)
    # print("Reward after latency penalty:", reward)
    # print("Latency info:", info["latency"])

By video prefetching: {62: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 69: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 67: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 56: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 55: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 45: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 14: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 41: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 18: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 79: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}
User requests: [{'seq': 0, 'u': 0, 'p': 0, 'video': 62, 'tiles': [{'tile_id': 5, 'size': 983040.0, 'events': {'alpha_p_u': 0, 'alpha_M_u': 0, 'alpha_C_u': 1, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile_id': 9, 'size': 983040.0, 'events': {'alpha_p_u': 0, 'alpha_M_u': 0, 'alpha_C_u': 1, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile_id': 6, 'size': 983040.0, 'events': {'alpha_p_